<a href="https://colab.research.google.com/github/himanshugithub360/PyTroch/blob/main/Training_Pipeline_using_Dataset_and_DataLoader.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [18]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

In [19]:
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [20]:
df.drop(columns=['id', 'Unnamed: 32'], inplace= True)

In [21]:
X_train, X_test, y_train, y_test = train_test_split(df.iloc[:, 1:], df.iloc[:, 0], test_size=0.2)

In [22]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [23]:
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

In [24]:
X_train_tensor = torch.from_numpy(X_train.astype(np.float32))
X_test_tensor = torch.from_numpy(X_test.astype(np.float32))
y_train_tensor = torch.from_numpy(y_train.astype(np.float32))
y_test_tensor = torch.from_numpy(y_test.astype(np.float32))

In [25]:
X_train_tensor.shape

torch.Size([455, 30])

In [26]:
from torch.utils.data import Dataset, DataLoader

class CustomDataset(Dataset):

  def __init__(self, features, labels):
    self.features = features
    self.labels = labels

  def __len__(self):
    return len(self.features)

  def __getitem__(self, idx):
    return self.features[idx], self.labels[idx]

In [27]:
train_dataset = CustomDataset(X_train_tensor, y_train_tensor)
test_dataset = CustomDataset(X_test_tensor, y_test_tensor)

In [28]:
train_dataset[1]

(tensor([ 0.3617, -0.4922,  0.4882,  0.1919,  2.6360,  2.3258,  1.9351,  1.9688,
          2.1541,  1.8731,  0.9866, -0.3100,  0.6510,  0.4727,  1.0772,  1.1257,
          0.5529,  1.5445, -0.2076,  1.7760,  0.3211, -0.5221,  0.3957,  0.1005,
          1.9682,  1.2478,  0.8009,  1.6695,  0.5612,  1.9795]),
 tensor(1.))

In [29]:
train_loader = DataLoader(train_dataset, batch_size = 32, shuffle = True)
test_loader = DataLoader(test_dataset, batch_size = 32, shuffle = True)

# Defining the model

In [34]:
import torch.nn as nn

class MySimpleNN(nn.Module):

  def __init__(self, num_features):

    super().__init__()
    self.linear = nn.Linear(num_features, 1)
    self.sigmoid = nn.Sigmoid()

  def forward(self, features):
    out = self.linear(features)
    out = self.sigmoid(out)

    return out

## Important Parameters

In [31]:
learning_rate = 0.1
epochs = 25



In [35]:
# create model
model = MySimpleNN(X_train_tensor.shape[1])

# define optimizer
optimizer = torch.optim.SGD(model.parameters(), lr = learning_rate)

# define loss function
loss_function = nn.BCELoss()


## Training Pipeline

In [37]:
# define loop
for epoch in range(epochs):

  for batch_features, batch_labels in train_loader:

    # forward pass
    y_pred = model(batch_features)

    #loss calculate
    loss = loss_function(y_pred, batch_labels.view(-1,1))

    #clear graidents
    optimizer.zero_grad()

    # backward pass
    loss.backward()

    # parameters update
    optimizer.step()


    # print loss in each epoch
    print(f'Epoch: {epoch + 1}, Loss: {loss.item()}')

Epoch: 1, Loss: 0.09105692058801651
Epoch: 1, Loss: 0.043854400515556335
Epoch: 1, Loss: 0.05617934465408325
Epoch: 1, Loss: 0.04019845649600029
Epoch: 1, Loss: 0.08165387809276581
Epoch: 1, Loss: 0.03295108675956726
Epoch: 1, Loss: 0.039329592138528824
Epoch: 1, Loss: 0.18417470157146454
Epoch: 1, Loss: 0.09263491630554199
Epoch: 1, Loss: 0.06999377906322479
Epoch: 1, Loss: 0.06259149312973022
Epoch: 1, Loss: 0.0675293579697609
Epoch: 1, Loss: 0.0707176923751831
Epoch: 1, Loss: 0.09508419781923294
Epoch: 1, Loss: 0.08071032911539078
Epoch: 2, Loss: 0.11695656180381775
Epoch: 2, Loss: 0.06864000111818314
Epoch: 2, Loss: 0.04631391912698746
Epoch: 2, Loss: 0.10648806393146515
Epoch: 2, Loss: 0.029228566214442253
Epoch: 2, Loss: 0.10291861742734909
Epoch: 2, Loss: 0.1968468427658081
Epoch: 2, Loss: 0.11212629824876785
Epoch: 2, Loss: 0.029226526618003845
Epoch: 2, Loss: 0.06870810687541962
Epoch: 2, Loss: 0.03863707184791565
Epoch: 2, Loss: 0.05494766682386398
Epoch: 2, Loss: 0.038998156

In [38]:
# Model evaluation using test_loader
model.eval()  # Set the model to evaluation mode
accuracy_list = []

with torch.no_grad():
    for batch_features, batch_labels in test_loader:
        # Forward pass
        y_pred = model(batch_features)
        y_pred = (y_pred > 0.8).float()  # Convert probabilities to binary predictions

        # Calculate accuracy for the current batch
        batch_accuracy = (y_pred.view(-1) == batch_labels).float().mean().item()
        accuracy_list.append(batch_accuracy)

# Calculate overall accuracy
overall_accuracy = sum(accuracy_list) / len(accuracy_list)
print(f'Accuracy: {overall_accuracy:.4f}')

Accuracy: 0.9783
